# Online Student Engagement Detection - Full GPU Training on Google Colab

This notebook is prepared for **FULL GPU TRAINING** of the **ST-GCN**, **Bi-GRU**, and **ConvNeXt1D** models on Google Colab using a **T4 GPU** runtime.

### Pre-requisites:
1. **Zip your processed dataset**: Compress your local `processed/` directory (containing all `.npy` files and `labels.csv`) into a single file named `processed.zip`.
2. **Upload to Google Drive**: Upload the `processed.zip` to your Google Drive in a folder named `student-engagement` (i.e. `/My Drive/student-engagement/processed.zip`).

## Step 1: Enable GPU Acceleration

Before running any cells, ensure that you have connected to a GPU runtime:
1. Go to **Runtime** > **Change runtime type**.
2. Select **T4 GPU** (or any available GPU) under *Hardware accelerator*.
3. Click **Save**.

## Step 2: Clone Repository & Install Dependencies

In [ ]:
# Clone your github repository
!git clone https://github.com/Shivani367/student-engagement.git
%cd student-engagement

In [ ]:
# Install all requirements automatically
!pip install -r requirements.txt
!pip install mediapipe pandas tqdm matplotlib scikit-learn

## Step 3: Mount Google Drive and Extract Processed Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy the zipped dataset from Google Drive and extract it
!cp /content/drive/MyDrive/student-engagement/processed.zip .
!unzip -q processed.zip -d .

# Verify that the processed files are extracted correctly
import os
print("Check labels.csv exists:", os.path.exists("processed/labels.csv"))
print("Number of preprocessed landmark files:", len(os.listdir("processed")) - 1)

## Step 4: Run Training on T4 GPU

We will train all three models. Training is optimized for the **T4 GPU** runtime by setting `--num_workers 2` (parallel CPU data loaders to speed up reading 5,300 files) and using optimized batch sizes. 

Checkpoints are saved to `checkpoints/` locally and backed up to Google Drive automatically after training.

### 1. Train ST-GCN Model
Trains the Spatial-Temporal Graph Convolutional Network on the full dataset.

In [ ]:
!python train_stgcn.py --epochs 15 --batch_size 32 --num_workers 2

### 2. Train Bi-GRU Model
Trains the Recurrent Neural Network baseline on the full dataset.

In [ ]:
!python train_bigru.py --epochs 15 --batch_size 128 --num_workers 2

### 3. Train ConvNeXt1D Model
Trains the Convolutional Neural Network baseline on the full dataset.

In [ ]:
!python train_convnext.py --epochs 15 --batch_size 32 --num_workers 2

## Step 5: Evaluate Models and Ensemble soft voting

We evaluate the individual models and the soft voting ensemble model. This generates accuracy, precision, recall, and F1 reports, and saves confusion matrix plots under `evaluation_results/`.

In [ ]:
# Run evaluation script
!python evaluate.py

In [ ]:
# Display confusion matrices, learning curves, and final report
import os
from IPython.display import Image, display

results_dir = 'evaluation_results'
if os.path.exists(results_dir):
    # 1. Display Confusion Matrices
    print("=== CONFUSION MATRICES ===\n")
    for f in sorted(os.listdir(results_dir)):
        if f.endswith('_cm.png'):
            print(f"--- Confusion Matrix: {f} ---")
            display(Image(os.path.join(results_dir, f)))
            print()
            
    # 2. Display Learning Curves
    print("\n=== LEARNING CURVES ===\n")
    for f in sorted(os.listdir(results_dir)):
        if f.endswith('_curves.png'):
            print(f"--- Learning Curves: {f} ---")
            display(Image(os.path.join(results_dir, f)))
            print()
            
    # 3. Display Final Evaluation Report
    print("\n=== FINAL EVALUATION REPORT ===\n")
    report_path = os.path.join(results_dir, 'final_evaluation_report.md')
    if os.path.exists(report_path):
        with open(report_path, 'r') as r_file:
            print(r_file.read())
    else:
        print("Report not found.")

## Step 6: Backup Checkpoints & Results to Google Drive

Saves all best checkpoints and evaluation plots/outputs to a results folder on Google Drive (`student-engagement/results/`) for persistence.

In [ ]:
# Create results backup directory on Drive
!mkdir -p /content/drive/MyDrive/student-engagement/results/checkpoints
!mkdir -p /content/drive/MyDrive/student-engagement/results/evaluation_results

# Copy checkpoints and evaluation results to Drive
!cp -r checkpoints/* /content/drive/MyDrive/student-engagement/results/checkpoints/
!cp -r evaluation_results/* /content/drive/MyDrive/student-engagement/results/evaluation_results/
print("All checkpoints and evaluation results successfully backed up to Google Drive!")

## 🔄 How to Resume Training if Colab Disconnects

If your Google Colab session gets disconnected (e.g., due to timeout, inactivity, or runtime crash), follow these steps to resume training from the last saved checkpoint on Google Drive:

1. **Re-connect** to a new T4 GPU runtime.
2. **Run Step 2 and Step 3** above to clone the repo, install requirements, mount Google Drive, and extract the dataset.
3. **Copy the saved checkpoint** from Google Drive back to your Colab workspace checkpoints folder:
   ```bash
   !mkdir -p checkpoints
   !cp /content/drive/MyDrive/student-engagement/results/checkpoints/stgcn_best.pth checkpoints/stgcn_best.pth
   ```
4. **Resume training** by running the training script with the `--resume` flag pointing to the copied checkpoint. Training will automatically pick up from the saved epoch and restore the optimizer states:

   * **ST-GCN**: `!python train_stgcn.py --epochs 15 --batch_size 32 --num_workers 2 --resume checkpoints/stgcn_best.pth`
   * **Bi-GRU**: `!python train_bigru.py --epochs 15 --batch_size 128 --num_workers 2 --resume checkpoints/bigru_best.pth`
   * **ConvNeXt1D**: `!python train_convnext.py --epochs 15 --batch_size 32 --num_workers 2 --resume checkpoints/convnext_best.pth`